In [ ]:
# ------------------------------------------------------------------------------
# 0. INSTALLATION PACKAGES & LIBRAIRIES ----
# ------------------------------------------------------------------------------
options(timeout = 3600)

system("conda install -c conda-forge libgdal-hdf5")
system("conda install -y r-terra=1.8_42 r-sf=1.0_20 r-ncdf4 r-fields")

library(terra)
library(sf)
library(dplyr)
library(ncdf4)
library(fields)
library(jsonlite)

# ------------------------------------------------------------------------------
# 1. PARAMÈTRES ET INPUTS ----
# ------------------------------------------------------------------------------

json_path <- "galaxy_inputs/galaxy_inputs.json"
json_data <- fromJSON(json_path)
file_copernicus <- json_data$Copernicus$path 
subarea <- json_data$subarea       
area_list <- trimws(unlist(strsplit(subarea, ",")))
m <- json_data$multiplier
index_max <- json_data$index_max    

In [ ]:
# ------------------------------------------------------------------------------
# 2. TÉLÉCHARGEMENT ET CHARGEMENT DES DONNÉES ----
# ------------------------------------------------------------------------------

# Télécharger et charger le fichier NetCDF
nc_file <- nc_open(file_copernicus)
nc_file_raw <- ncvar_get(nc_file,"CHL")
lat_vals <- ncvar_get(nc_file, "latitude")
lon_vals <- ncvar_get(nc_file, "longitude")
nc_close(nc_file)

# Inverser les colonnes (bug connu)
nc_file_raw <- nc_file_raw[, ncol(nc_file_raw):1]

# Créer raster à partir du NetCDF
r <- rast(nrows = length(lat_vals),
          ncols = length(lon_vals),
          xmin = min(lon_vals),
          xmax = max(lon_vals),
          ymin = min(lat_vals),
          ymax = max(lat_vals),
          crs = "EPSG:4326")
values(r) <- as.vector(nc_file_raw)

# Reprojection
raster_project <- project(r, "EPSG:6932")

# Télécharger GEBCO
download.file("https://gis.ccamlr.org/geoserver/www/GEBCO2024_5000.tif", 
              "GEBCO2024_5000.tif", mode = "wb")
gebco_raster <- rast("GEBCO2024_5000.tif")

# ------------------------------------------------------------------------------
# 3. TÉLÉCHARGER ET FILTRER LES SOUS-ZONES ASD ----
# ------------------------------------------------------------------------------

output_dir_asd <- "asd"
dir.create(output_dir_asd, showWarnings = FALSE)

urls_asd <- c(
  "https://raw.githubusercontent.com/ccamlr/data/refs/tags/v0.5.0/geographical_data/asd/asd-shapefile-EPSG6932.shp",
  "https://raw.githubusercontent.com/ccamlr/data/refs/tags/v0.5.0/geographical_data/asd/asd-shapefile-EPSG6932.shx",
  "https://raw.githubusercontent.com/ccamlr/data/refs/tags/v0.5.0/geographical_data/asd/asd-shapefile-EPSG6932.dbf",
  "https://raw.githubusercontent.com/ccamlr/data/refs/tags/v0.5.0/geographical_data/asd/asd-shapefile-EPSG6932.prj"
)
destfiles_asd <- file.path(output_dir_asd, basename(urls_asd))
for (i in seq_along(urls_asd)) {
  download.file(urls_asd[i], destfiles_asd[i], mode = "wb")
}

# Lire le shapefile et filtrer les sous-zones
ASDs <- st_read(file.path(output_dir_asd, "asd-shapefile-EPSG6932.shp"), quiet = TRUE)
subareas_selected <- ASDs %>% filter(GAR_Short_ %in% area_list)
subareas_vect <- vect(subareas_selected)

# ------------------------------------------------------------------------------
# 4. CALCUL TAILLE DE L'IMAGE EN FONCTION DE LA ZONE ----
# ------------------------------------------------------------------------------

# Juste pour calculer l'étendue de la zone sélectionnée
raster_cropped <- crop(raster_project, subareas_vect)
raster_masked <- mask(raster_cropped, subareas_vect)

# Cropper et masker GEBCO aussi
gebco_cropped <- crop(gebco_raster, subareas_vect)
gebco_masked <- mask(gebco_cropped, subareas_vect)

raster_extent <- ext(raster_masked)
w <- abs(raster_extent[2] - raster_extent[1]) / 10000
h <- abs(raster_extent[4] - raster_extent[3]) / 10000

# ------------------------------------------------------------------------------
# 5. AFFICHAGE FINAL ET EXPORT PNG ----
# ------------------------------------------------------------------------------

png(filename = "outputs/Fig9.png", width = w * m, height = h * m)


# Palette pour chlorophylle
custom_palette <- colorRampPalette(c(
  "#0286c4", "#1698b7", "#33a1b8", "#4bafad", "#6abbb4",
  "#82cca9", "#94dba1", "#a5e49d", "#b6eba3", "#d6f1ac",
  "#e2f9ac", "#f1feb8", "#fffdb3", "#fff2a4", "#ffe493",
  "#fed48a", "#ffcb6e", "#fcbc65", "#f8b253", "#ff9645",
  "#ff642c", "#f54d29", "#f82619", "#f40616"
))(1000)
custom_palette <- c(custom_palette, "red")
breaks <- c(seq(0.03, index_max, length.out = 100), Inf)

# Couleurs pour GEBCO
my_colors <- c("white", "grey")
my_breaks <- c(-Inf, 0, Inf)


# Crée un layout avec 2 lignes et 1 colonne : une pour la carte et une pour la légende
layout(matrix(c(1, 2), 2, 1), heights = c(4, 1))  # 4 parts pour la carte, 1 part pour la légende

# Zone 1 : Affichage de la carte
par(mar = c(0, 0, 0, 0))  # Marges ajustées pour ne pas avoir de marges inutiles
plot(gebco_masked, col = my_colors, breaks = my_breaks, legend = FALSE, axes = FALSE, box = FALSE)

# Affichage du raster CHL
plot(raster_masked, col = custom_palette, breaks = breaks, legend = FALSE, add = TRUE)

# Ajout des limites des sous-zones en rouge
plot(subareas_vect, add = TRUE, lwd = 0.75*m, border = 'black')

# Ajout des centroids
centroids <- st_centroid(st_geometry(subareas_selected))
text(st_coordinates(centroids), labels = subareas_selected$GAR_Long_L, col = "black", cex = 1*m)

# Zone 2 : Affichage de la légende en bas
par(mar = c(3, 0, 0, 0))  # Marges ajustées pour la légende en bas
zlim <- c(min(values(raster_project), na.rm = TRUE), 5)

# Affichage de la légende en bas, de manière horizontale
image.plot(
    raster_masked, 
    zlim = zlim,
    col = custom_palette,
    legend.only = TRUE,
    horizontal = TRUE,  # Légende horizontale
    legend.width = 1,  # Largeur de la légende
    legend.shrink = 0.7,
    axis.args = list(cex.axis = 0.8, col.axis = "black", at = c(0.03, 1, 2, 3, 4, 5)) # Contrôle de l'axe de légende
)
# Rétablir le layout par défaut
layout(1)  # Réinitialiser le layout pour revenir au comportement normal


dev.off()


